# To use on google colab

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import sys
sys.path.append('/content/drive/MyDrive/RI-Project-trying-to-start-phase-2/1st Stage/NLPSearch-phase1/')


In [3]:
%cd "/content/drive/MyDrive/RI-Project-trying-to-start-phase-2/1st Stage/NLPSearch-phase1/"

/content/drive/MyDrive/RI-Project-trying-to-start-phase-2/1st Stage/NLPSearch-phase1


In [4]:
!ls /content/drive/MyDrive/RI-Project-trying-to-start-phase-2/'1st Stage'/NLPSearch-phase1/*.py

'/content/drive/MyDrive/RI-Project-trying-to-start-phase-2/1st Stage/NLPSearch-phase1/Memory.py'
'/content/drive/MyDrive/RI-Project-trying-to-start-phase-2/1st Stage/NLPSearch-phase1/OpenSearchSimpleAPI.py'
'/content/drive/MyDrive/RI-Project-trying-to-start-phase-2/1st Stage/NLPSearch-phase1/rank_metric.py'
'/content/drive/MyDrive/RI-Project-trying-to-start-phase-2/1st Stage/NLPSearch-phase1/rank_with_history.py'
'/content/drive/MyDrive/RI-Project-trying-to-start-phase-2/1st Stage/NLPSearch-phase1/ReRank.py'
'/content/drive/MyDrive/RI-Project-trying-to-start-phase-2/1st Stage/NLPSearch-phase1/TRECCASTeval.py'


In [5]:
!pip install opensearch-py
!pip install numpy
!pip install transformers
!pip install torch
!pip install sklearn

  Using cached sklearn-0.0.post12.tar.gz (2.6 kB)
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.




---



#Passage Re-Ranking with Natural Language Embedding

In [6]:
from importlib import reload
import TRECCASTeval as trec
import OpenSearchSimpleAPI as osearch

osearch = reload(osearch)

test_bed = trec.ConvSearchEvaluation()
opensearch = osearch.OSsimpleAPI()


In [7]:
conv_topic_idx = 0

In [8]:
test_bed.test_relevance_judgments

,topic_turn_id,dummy,docid,rel
0,31_1,Q0,CAR_116d829c4c800c2fc70f11692fec5e8c7e975250,0
1,31_1,Q0,CAR_1463f964653c5c9f614a0a88d26b175e4a8120f1,1
2,31_1,Q0,CAR_172e16e89ea3d5546e53384a27c3be299bcfe968,2
3,31_1,Q0,CAR_1c93ef499a0c2856c4a857b0cb4720c380dda476,0
4,31_1,Q0,CAR_2174ad0aa50712ff24035c23f59a3c2b43267650,3
...,...,...,...,...
29345,79_9,Q0,MARCO_8795229,0
29346,79_9,Q0,MARCO_8795231,0
29347,79_9,Q0,MARCO_8795233,0
29348,79_9,Q0,MARCO_8795236,0


## Bulding triplets

In [115]:
from opensearchpy import NotFoundError


def get_relevances_judgement_of_turn(relevance_judgments, turn_id):
    filter = relevance_judgments['topic_turn_id'].map(lambda x: x == turn_id)
    return relevance_judgments[filter]

def get_rel_and_doc_body(query,relevance_judgments, turn_id):
    relevances = get_relevances_judgement_of_turn(relevance_judgments,turn_id)

    triplets = []
    for row in relevances.values:
        doc_id = row[2]
        try:
            doc_body = opensearch.get_doc_body(doc_id)
        except:
            continue
        if(row[3] > 0):
          rel = 1
        else:
          rel = 0

        if(doc_body == None):
            continue

        triplets.append([query, doc_body, rel])

    return triplets

In [116]:
triplets = []
queries = []

for topic in test_bed.train_topics:
    conv_id = topic['number']

    if conv_id not in (1, 2, 4, 7, 15, 17,18,22,23,24,25,27,30):
        continue

    print()
    print(conv_id, "  ", topic['title'])

    for turn in topic['turn']:
        turn_id = turn['number']
        query = turn['raw_utterance']
        topic_turn_id = '%d_%d'% (conv_id, turn_id)

        query_with_topic = topic['title'] + " " + query
        queries.append([query_with_topic, topic_turn_id])

        print(topic_turn_id, query)
        rel_and_docs = get_rel_and_doc_body(query_with_topic, test_bed.train_relevance_judgments, topic_turn_id)
        if(rel_and_docs == None):
            continue

        triplets.extend(rel_and_docs)


1    Career choice for Nursing and Physician's Assistant
1_1 What is a physician's assistant?
1_2 What are the educational requirements required to become one?
1_3 What does it cost?
1_4 What's the average starting salary in the UK?
1_5 What about in the US?
1_6 What school subjects are needed to become a registered nurse?
1_7 What is the PA average salary vs an RN?
1_8 What the difference between a PA and a nurse practitioner?
1_9 Do NPs or PAs make more?
1_10 Is a PA above a NP?
1_11 What is the fastest way to become a NP?
1_12 How much longer does it take to become a doctor after being an NP?



2    Goat breeds
2_1 What are the main breeds of goat?
2_2 Tell me about boer goats.
2_3 What breed is good for meat?
2_4 Are angora goats good for it?


2_5 What about boer goats?
2_6 What are pygmies used for?


2_7 What is the best for fiber production?


2_8 How long do Angora goats live?
2_9 Can you milk them?


2_10 How many can you have per acre?


... (26 outputs left out)

30_3 How does it compare to Windows?


30_4 Which of these is more popular?


30_5 How do I install software on it?


30_6 What is an easy way to install Python?
30_7 Tell me about how I can share files.


In [117]:
len(triplets)

2352

In [118]:
print(queries)

[["Career choice for Nursing and Physician's Assistant What is a physician's assistant?", '1_1'], ["Career choice for Nursing and Physician's Assistant What are the educational requirements required to become one?", '1_2'], ["Career choice for Nursing and Physician's Assistant What does it cost?", '1_3'], ["Career choice for Nursing and Physician's Assistant What's the average starting salary in the UK?", '1_4'], ["Career choice for Nursing and Physician's Assistant What about in the US?", '1_5'], ["Career choice for Nursing and Physician's Assistant What school subjects are needed to become a registered nurse?", '1_6'], ["Career choice for Nursing and Physician's Assistant What is the PA average salary vs an RN?", '1_7'], ["Career choice for Nursing and Physician's Assistant What the difference between a PA and a nurse practitioner?", '1_8'], ["Career choice for Nursing and Physician's Assistant Do NPs or PAs make more?", '1_9'], ["Career choice for Nursing and Physician's Assistant I

### Saving triplets in files

In [119]:
import csv

In [120]:
with open('triplets_3.csv', 'w') as file:
    writer = csv.writer(file)
    writer.writerows(triplets)

In [121]:
with open('queries_3.csv', 'w') as file:
    writer = csv.writer(file)
    writer.writerows(queries)

In [ ]:
# Load data from a CSV file
with open('output.csv', 'r') as file:
    reader = csv.reader(file)  # Use csv.reader to read rows
    data = []
    for row in reader:
        # Convert numeric values back to integers if possible
        processed_row = [int(x) if x.isdigit() else x for x in row]
        data.append(processed_row)

print(data)
triplets = data

[["[CLS]Career choice for Nursing and Physician's Assistant What is a physician's assistant?[SEP]Physician assistants work under the supervision of a physician or surgeon; however, their specific duties and the extent to which they must be supervised differ from state to state. Physician assistants work in all areas of medicine, including primary care and family medicine, emergency medicine, and psychiatry.The work of physician assistants depends in large part on their specialty and what their supervising physician needs them to do.For example, a physician assistant working in surgery may close incisions and provide care before and after the operation.he work of physician assistants depends in large part on their specialty and what their supervising physician needs them to do. For example, a physician assistant working in surgery may close incisions and provide care before and after the operation.", 1], ["[CLS]Career choice for Nursing and Physician's Assistant What is a physician's as

## Extracting CLS embeddings

In [122]:
import numpy as np
import torch
from transformers import BertTokenizer,BertModel,AutoTokenizer,AutoConfig,AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score


In [123]:
model_path = 'bert-base-uncased'

In [124]:
tokenizer = AutoTokenizer.from_pretrained(model_path)
config = AutoConfig.from_pretrained(model_path,  output_hidden_states=True, output_attentions=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModel.from_pretrained(model_path, config=config)
model.to(device)
device

device(type='cuda')

In [125]:
# Function to extract CLS embeddings
def extract_cls(query_pairs, embeddings, batch_size=32):
    for batch_idx in range(0, len(query_pairs), batch_size):
        print(batch_idx)
        batch_data = query_pairs[batch_idx:batch_idx + batch_size]
        inputs = tokenizer.batch_encode_plus(batch_data,
                                             return_tensors='pt',
                                             add_special_tokens=True,
                                             max_length=512,
                                             truncation=True,
                                             padding=True)
        with torch.no_grad():
            inputs.to(device)
            outputs = model(**inputs)

        batch_cls = outputs['hidden_states'][-1][:,0,:]

        batch_cls = torch.nn.functional.normalize(batch_cls, p=2, dim=1)  # L2 normalize

        embeddings[batch_idx:batch_idx + batch_size] = batch_cls.cpu()  # Store embeddings

        # Clear temporary variables
        del inputs, outputs, batch_cls
        torch.cuda.empty_cache()
    return embeddings

In [128]:
# Prepare embeddings and relevance labels
train_embeddings = np.zeros((len(triplets), 768))  # 768 is the size of BERT's CLS vector
relevance_labels = [pair[2] for pair in triplets]  # Extract relevance scores

# Extract CLS embeddings for each query-passage pair
train_embeddings = extract_cls([[pair[0],pair[1]] for pair in triplets], embeddings=train_embeddings, batch_size=32)

0
32
64
96
128
160
192
224
256
288
320
352
384
416
448
480
512
544
576
608
640
672
704
736
768
800
832
864
896
928
960
992
1024
1056
1088
1120
1152
1184
1216
1248
1280
1312
1344
1376
1408
1440
1472
1504
1536
1568
1600
1632
1664
1696
1728
1760
1792
1824
1856
1888
1920
1952
1984
2016
2048
2080
2112
2144
2176
2208
2240
2272
2304
2336


## Logistic Model Training

In [129]:
X_train, X_test, y_train, y_test = train_test_split(train_embeddings,relevance_labels, test_size=0.2, random_state=42)

In [130]:
logistic_regressor = LogisticRegression()
logistic_regressor.fit(X_train, y_train)

LogisticRegression()

In [131]:
y_pred = logistic_regressor.predict(X_train)
y_pred_probs = logistic_regressor.predict_proba(X_train)[:, 1]

In [132]:
# Evaluate the model
print("Training results")
print("Accuracy:", accuracy_score(y_train, y_pred))
print("\nClassification Report:\n", classification_report(y_train, y_pred))

Training results
Accuracy: 0.7363104731525785

Classification Report:
               precision    recall  f1-score   support

           0       0.74      0.99      0.85      1378
           1       0.62      0.04      0.07       503

    accuracy                           0.74      1881
   macro avg       0.68      0.51      0.46      1881
weighted avg       0.71      0.74      0.64      1881



In [133]:
y_pred = logistic_regressor.predict(X_test)
y_pred_probs = logistic_regressor.predict_proba(X_test)[:, 1]

In [134]:
# Evaluate the model
print("Testing results")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Testing results
Accuracy: 0.721868365180467

Classification Report:
               precision    recall  f1-score   support

           0       0.72      0.99      0.84       337
           1       0.71      0.04      0.07       134

    accuracy                           0.72       471
   macro avg       0.72      0.52      0.45       471
weighted avg       0.72      0.72      0.62       471



# Model Inference

## Building (query, document) pairs for top 100 documents for each query

In [135]:
query_passage_pairs = []

for query in queries:
    print(query)
    try:
      passages = opensearch.search_body(query[0], numDocs=100)["_source.contents"].values
    except:
      continue
    for passage in passages:
        query_passage_pairs.append([query[0],passage,query[1]])

["Career choice for Nursing and Physician's Assistant What is a physician's assistant?", '1_1']
["Career choice for Nursing and Physician's Assistant What are the educational requirements required to become one?", '1_2']
["Career choice for Nursing and Physician's Assistant What does it cost?", '1_3']
["Career choice for Nursing and Physician's Assistant What's the average starting salary in the UK?", '1_4']
["Career choice for Nursing and Physician's Assistant What about in the US?", '1_5']
["Career choice for Nursing and Physician's Assistant What school subjects are needed to become a registered nurse?", '1_6']
["Career choice for Nursing and Physician's Assistant What is the PA average salary vs an RN?", '1_7']
["Career choice for Nursing and Physician's Assistant What the difference between a PA and a nurse practitioner?", '1_8']
["Career choice for Nursing and Physician's Assistant Do NPs or PAs make more?", '1_9']
["Career choice for Nursing and Physician's Assistant Is a PA abo

In [136]:
query_passage_pairs

[["Career choice for Nursing and Physician's Assistant What is a physician's assistant?",
  "Salary for Physician Assistants. Also known as: Anesthesiologist Assistant, Certified Physician's Assistant, Family Practice Physician Assistant, Orthopaedic Physician Assistant, Orthopedic Physician Assistant, Pediatric Physician Assistant, Radiology Practitioner Assistant, Surgical Physician Assistant.",
  '1_1'],
 ["Career choice for Nursing and Physician's Assistant What is a physician's assistant?",
  "Salary for Anesthesiologist Assistants. Also known as: Anesthesiologist Assistant, Certified Physician's Assistant, Family Practice Physician Assistant, Orthopaedic Physician Assistant, Orthopedic Physician Assistant, Pediatric Physician Assistant, Radiology Practitioner Assistant, Surgical Physician Assistant.",
  '1_1'],
 ["Career choice for Nursing and Physician's Assistant What is a physician's assistant?",
  "What is the difference between a medical assistant and physician assistant? Me

In [137]:
len(query_passage_pairs)

12000

In [138]:
with open('query_passage_pairs_3.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerows(query_passage_pairs)

## Extracing CLS embeddings

In [139]:
# Extract CLS embeddings for all query-passage pairs
query_passage_pairs_without_turn_id = [[row[0], row[1]] for row in query_passage_pairs]

inference_embeddings = np.zeros((len(query_passage_pairs), 768))  # Assuming 768 dimensions for BERT embeddings
inference_embeddings = extract_cls(query_passage_pairs_without_turn_id, embeddings=inference_embeddings, batch_size=32)

0
32
64
96
128
160
192
224
256
288
320
352
384
416
448
480
512
544
576
608
640
672
704
736
768
800
832
864
896
928
960
992
1024
1056
1088
1120
1152
1184
1216
1248
1280
1312
1344
1376
1408
1440
1472
1504
1536
1568
1600
1632
1664
1696
1728
1760
1792
1824
1856
1888
1920
1952
1984
2016
2048
2080
2112
2144
2176
2208
2240
2272
2304
2336
2368
2400
2432
2464
2496
2528
2560
2592
2624
2656
2688
2720
2752
2784
2816
2848
2880
2912
2944
2976
3008
3040
3072
3104
3136
3168
3200
3232
3264
3296
3328
3360
3392
3424
3456
3488
3520
3552
3584
3616
3648
3680
3712
3744
3776
3808
3840
3872
3904
3936
3968
4000
4032
4064
4096
4128
4160
4192
4224
4256
4288
4320
4352
4384
4416
4448
4480
4512
4544
4576
4608
4640
4672
4704
4736
4768
4800
4832
4864
4896
4928
4960
4992
5024
5056
5088
5120
5152
5184
5216
5248
5280
5312
5344
5376
5408
5440
5472
5504
5536
5568
5600
5632
5664
5696
5728
5760
5792
5824
5856
5888
5920
5952
5984
6016
6048
6080
6112
6144
6176
6208
6240
6272
6304
6336
6368
6400
6432
6464
6496
6528
6560
6592
66

In [69]:
print(inference_embeddings)

[[-0.06031382  0.02217891 -0.02856671 ... -0.02542698  0.01833249
   0.04322433]
 [-0.05286395  0.01955962 -0.02831251 ... -0.02534507  0.01823561
   0.04792405]
 [-0.0521942   0.03427834 -0.03684151 ... -0.03400815  0.03707785
   0.02687828]
 ...
 [-0.02812606  0.00709658  0.00576624 ... -0.02851035  0.03369816
   0.03350361]
 [-0.02213729  0.00478184  0.00436731 ... -0.03861387  0.02750824
   0.04276343]
 [-0.05198935 -0.01038694 -0.00450632 ... -0.03790107  0.00717684
   0.03387095]]


In [70]:
with open('embeddings_query_passage_pairs_3.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerows(inference_embeddings)

## Using the model to predict the probability of the relevance

In [140]:
# Predict relevance probabilities
predicted_scores = logistic_regressor.predict_proba(inference_embeddings)[:, 1]  # Probability for class 1

In [141]:
print(predicted_scores)
print(query_passage_pairs)

[0.33358773 0.29453127 0.39718564 ... 0.06809854 0.08608611 0.31497015]
[["Career choice for Nursing and Physician's Assistant What is a physician's assistant?", "Salary for Physician Assistants. Also known as: Anesthesiologist Assistant, Certified Physician's Assistant, Family Practice Physician Assistant, Orthopaedic Physician Assistant, Orthopedic Physician Assistant, Pediatric Physician Assistant, Radiology Practitioner Assistant, Surgical Physician Assistant.", '1_1'], ["Career choice for Nursing and Physician's Assistant What is a physician's assistant?", "Salary for Anesthesiologist Assistants. Also known as: Anesthesiologist Assistant, Certified Physician's Assistant, Family Practice Physician Assistant, Orthopaedic Physician Assistant, Orthopedic Physician Assistant, Pediatric Physician Assistant, Radiology Practitioner Assistant, Surgical Physician Assistant.", '1_1'], ["Career choice for Nursing and Physician's Assistant What is a physician's assistant?", "What is the differ

## Sorting and ordering each (query, document) pair

In [142]:
result_scores_per_query_passage = [[pair[0], pair[1], value] for pair,value in zip(query_passage_pairs, predicted_scores)]

In [143]:
from collections import defaultdict

# Group by query
query_passage_groups = defaultdict(list)
for query, passage, score in result_scores_per_query_passage:
  query_passage_groups[query].append([passage,score])

# Sort each group by score (descending)
for query in query_passage_groups:
    query_passage_groups[query] = sorted(query_passage_groups[query], key=lambda x: x[1], reverse=True)

In [144]:
result_scores_per_query_passage

[["Career choice for Nursing and Physician's Assistant What is a physician's assistant?",
  "Salary for Physician Assistants. Also known as: Anesthesiologist Assistant, Certified Physician's Assistant, Family Practice Physician Assistant, Orthopaedic Physician Assistant, Orthopedic Physician Assistant, Pediatric Physician Assistant, Radiology Practitioner Assistant, Surgical Physician Assistant.",
  0.33358773090675],
 ["Career choice for Nursing and Physician's Assistant What is a physician's assistant?",
  "Salary for Anesthesiologist Assistants. Also known as: Anesthesiologist Assistant, Certified Physician's Assistant, Family Practice Physician Assistant, Orthopaedic Physician Assistant, Orthopedic Physician Assistant, Pediatric Physician Assistant, Radiology Practitioner Assistant, Surgical Physician Assistant.",
  0.29453126707693494],
 ["Career choice for Nursing and Physician's Assistant What is a physician's assistant?",
  "What is the difference between a medical assistant an

In [145]:
result_scores_per_query_passage[0]

["Career choice for Nursing and Physician's Assistant What is a physician's assistant?",
 "Salary for Physician Assistants. Also known as: Anesthesiologist Assistant, Certified Physician's Assistant, Family Practice Physician Assistant, Orthopaedic Physician Assistant, Orthopedic Physician Assistant, Pediatric Physician Assistant, Radiology Practitioner Assistant, Surgical Physician Assistant.",
 0.33358773090675]

In [146]:
# Print the sorted results
for query, passages in query_passage_groups.items():
    max_docs_shown_per_query = 5
    i = 0
    print(f"Query: {query}")
    for passage, score in passages:
        if(i >= max_docs_shown_per_query):
          i = 0
          break;
        print(f"\t- Passage: {passage}")
        print(f"\t  Score: {score}")

        i = i + 1

Query: Career choice for Nursing and Physician's Assistant What is a physician's assistant?
	- Passage: Physician Assistant. A physician assistant is a person who has successfully completed an accredited education program for physician assistant, is licensed by the state and is practicing within the scope of that license. Physician assistants are formally trained to perform many of the routine, time-consuming tasks a physician can do. In some states, they may prescribe medications. They take medical histories, perform physical exams, order lab tests and x-rays, and give inoculations. Most states require that they work under the supervision of a physician.
	  Score: 0.42355206042486454
	- Passage: Differences Between a Physician Assistant and a Nurse Practitioner. Both physician assistants and nurse practitioners provide health care, although the restrictions and requirements in each field vary. If you enjoy caring for others and are willing to commit to formal education, then a career 

In [147]:
opensearch.get_doc_body("MARCO_2334575")

'For example, the chemical energy in a battery is converted into electrical energy. The electrical energy, which involves the motion of electrical charges or currents, can be used to power everyday devices like computers and flashlights. Mechanical energy can also be transformed into electrical energy. For example, a hydroelectric plant uses the mechanical energy of flowing water to generate electrical energy. Wind energy is another mechanical to electrical energy transformation.'

In [148]:
sorted_data = sorted(result_scores_per_query_passage, key=lambda x: x[2], reverse=True)

i = 0

for row in sorted_data:
  if i >= 20:
      break
  print("Query: "+ row[0])
  print("Doc: "+row[1])
  print("Score: "+str(row[2]))
  print("-------")
  i = i+1

Query: Forms of energy Tell me about mechanical energy.
Doc: Mechanical Energy. Mechanical energy is the energy that is possessed by an object due to its motion or due to its position. Mechanical energy can be either kinetic energy (energy of motion) or potential energy (stored energy of position).echanical Energy as the Ability to Do Work An object that possesses mechanical energy is able to do work. In fact, mechanical energy is often defined as the ability to do work. Any object that possesses mechanical energy-whether it is in the form of potential energy or kinetic energy-is able to do work.
Score: 0.6007440627657298
-------
Query: Forms of energy Are potential and kinetic the same?
Doc: Types of Energy. There are two types of energy: Potential (or stored energy) Kinetic (or energy in motion) Kinetic Energy is energy in motion. Moving water and wind are examples of kinetic energy. Electricity is kinetic energy because, even though you can't see it happen, electricity involves elec

In [149]:
# Sort passages by predicted scores
sorted_indices = np.argsort(predicted_scores)[::-1]  # Descending order
sorted_passages = [query_passage_pairs[i] for i in sorted_indices]
# maybe tentar multiplar o sorted passages pelo predicted socres com o .dot e ver qual o resultado da matriz
i = 0;
#sorted_passages
for idx in sorted_indices:
  if i >= 100:
    break
  print(predicted_scores[idx])
  print("Query:"+query_passage_pairs[idx][0])
  print("Doc:"+query_passage_pairs[idx][1])
  print("----")
  i = i + 1

0.6007440627657298
Query:Forms of energy Tell me about mechanical energy.
Doc:Mechanical Energy. Mechanical energy is the energy that is possessed by an object due to its motion or due to its position. Mechanical energy can be either kinetic energy (energy of motion) or potential energy (stored energy of position).echanical Energy as the Ability to Do Work An object that possesses mechanical energy is able to do work. In fact, mechanical energy is often defined as the ability to do work. Any object that possesses mechanical energy-whether it is in the form of potential energy or kinetic energy-is able to do work.
----
0.5906132650960484
Query:Forms of energy Are potential and kinetic the same?
Doc:Types of Energy. There are two types of energy: Potential (or stored energy) Kinetic (or energy in motion) Kinetic Energy is energy in motion. Moving water and wind are examples of kinetic energy. Electricity is kinetic energy because, even though you can't see it happen, electricity involves

In [150]:
sorted_indices[-1]

4481